# 04 — Model Comparison: CNN vs BERT vs XLNet

Loads `outputs/*_metrics.json` written by notebooks 01–03 and compares the models on accuracy, precision, recall, F1-score, F1-Fake, ROC-AUC, the confusion matrices and per-category performance. Run notebooks 01–03 first.

In [ ]:
# --- Setup: mount Google Drive (dataset lives there, not in GitHub) ---
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/FakeReviewProject'   # change if your folder differs
DATA_DIR    = f'{PROJECT_DIR}/dataset'                     # train.csv, test.csv
OUT_DIR     = f'{PROJECT_DIR}/outputs'
os.makedirs(OUT_DIR, exist_ok=True)
print(os.listdir(DATA_DIR))

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve

MODELS = ['CNN', 'BERT', 'XLNet']
M = {m: json.load(open(f'{OUT_DIR}/{m.lower()}_metrics.json')) for m in MODELS if os.path.exists(f'{OUT_DIR}/{m.lower()}_metrics.json')}
cols = ['accuracy', 'precision', 'recall', 'f1_fake', 'f1_macro', 'roc_auc', 'threshold', 'train_size', 'train_time_s', 'infer_time_s']
table = pd.DataFrame({m: {c: M[m].get(c) for c in cols} for m in M}).T
table.to_csv(f'{OUT_DIR}/model_comparison.csv')
table.style.format(precision=4).highlight_max(subset=['accuracy', 'precision', 'recall', 'f1_fake', 'f1_macro', 'roc_auc'], color='#c6efce')

In [ ]:
metrics = ['accuracy', 'precision', 'recall', 'f1_fake', 'roc_auc']
ax = table[metrics].astype(float).plot.bar(figsize=(10, 4.5), rot=0, width=0.8)
ax.set_ylim(0, 1); ax.set_title('CNN vs BERT vs XLNet (test set)'); ax.legend(ncol=5, loc='lower center', bbox_to_anchor=(0.5, -0.3))
for c in ax.containers: ax.bar_label(c, fmt='%.2f', fontsize=7)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/comparison_bar.png', dpi=150); plt.show()

In [ ]:
test_y = pd.read_csv(f'{DATA_DIR}/test.csv').dropna(subset=['review_text', 'label']).label.astype(int).values
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for m in M:
    p = np.load(f'{OUT_DIR}/{m.lower()}_test_probs.npy')
    fpr, tpr, _ = roc_curve(test_y, p); ax[0].plot(fpr, tpr, label=f"{m} (AUC={M[m]['roc_auc']:.3f})")
    pr, rc, _ = precision_recall_curve(test_y, p); ax[1].plot(rc, pr, label=m)
ax[0].plot([0, 1], [0, 1], 'k--', lw=0.8); ax[0].set(title='ROC curve', xlabel='FPR', ylabel='TPR'); ax[0].legend()
ax[1].set(title='Precision–Recall (Fake class)', xlabel='Recall', ylabel='Precision'); ax[1].legend()
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/comparison_roc_pr.png', dpi=150); plt.show()

In [ ]:
fig, ax = plt.subplots(1, len(M), figsize=(4.2 * len(M), 3.6))
for a, m in zip(np.atleast_1d(ax), M):
    sns.heatmap(np.array(M[m]['confusion_matrix']), annot=True, fmt='d', cmap='Blues', cbar=False, ax=a,
                xticklabels=['Genuine', 'Fake'], yticklabels=['Genuine', 'Fake'])
    a.set(title=m, xlabel='Predicted', ylabel='Actual')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/comparison_confusion.png', dpi=150); plt.show()

In [ ]:
# Per-category F1-Fake (paper Table 6 layout)
per_cat = pd.DataFrame({m: {c: v['f1_fake'] for c, v in M[m]['per_category'].items()} for m in M})
per_cat.to_csv(f'{OUT_DIR}/comparison_per_category_f1.csv')
per_cat.plot.barh(figsize=(9, 4), title='F1-Fake by product category'); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/comparison_per_category.png', dpi=150); plt.show()
per_cat.round(4)

In [ ]:
best = table['f1_fake'].astype(float).idxmax()
print(f'Best model by F1-Fake: {best}  (F1-Fake={table.loc[best, "f1_fake"]:.4f}, ROC-AUC={table.loc[best, "roc_auc"]:.4f})')